# 馬券風 次単語予想モード — レースデータの作成

LLM が次の単語を決めるまでを競馬に見立てるための CSV を作ります。

- **出走馬** = 次の単語の候補（**どれを走らせるかは自分で選びます**）
- **着順** = 最終出力確率の順位
- **途中経過** = 各層で logit lens を掛けた確率（1層 = 100m）

**1回の実行で 1 レースぶん**を作ります。`races_01.csv`, `races_02.csv` … と
番号付きで書き出されるので、プロンプトを変えて何度か回してレースを溜めてください。
アプリ側は取り込んだレースの中から**ランダムに 5 つ選んで**遊ぶので、多いほど飽きません。

## 動かす場所

**ローカルでも Colab でも動きます。GPU は要りません**（CPU で十分な大きさのモデルを使います）。

- ローカル … [notebooks/README.md](README.md) の手順で仮想環境を作ってから開く
- Colab … そのまま実行（次のセルが自動で必要なものを入れます）

## 1. 準備

In [ ]:
# Colab のときだけ入れる。ローカルは requirements.txt で入れてある前提
import importlib.util, sys

# find_spec("google.colab") は google パッケージ自体が無いと例外を投げるので、素直に import で判定する
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install "torch>=2.2" "transformers>=4.44,<5" pandas matplotlib

missing = [m for m in ("torch", "transformers", "pandas", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise SystemExit(f"{', '.join(missing)} がありません。notebooks/README.md の手順で入れてください。")
print("Python", sys.version.split()[0], "on", "Colab" if IN_COLAB else "ローカル")

In [ ]:
import glob, os, re
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch", torch.__version__, "/ numpy", np.__version__)

def setup_japanese_font():
    """グラフの日本語が豆腐（□）にならないようにする。

    matplotlib の既定フォントには日本語が入っていないので、OS にあるものを探して使う。
    見つからなければ諦めて英字だけにする（グラフ自体は出る）。
    """
    have = {f.name for f in font_manager.fontManager.ttflist}
    for name in ("Hiragino Sans", "Hiragino Maru Gothic Pro", "Yu Gothic", "Meiryo",
                 "MS Gothic", "Noto Sans CJK JP", "Noto Sans JP", "IPAexGothic",
                 "IPAGothic", "TakaoGothic", "VL Gothic"):
        if name in have:
            matplotlib.rcParams["font.family"] = name
            matplotlib.rcParams["axes.unicode_minus"] = False
            return name
    return None

FONT = setup_japanese_font()
if FONT:
    print("日本語フォント:", FONT)
else:
    print("日本語フォントが見つかりません。グラフの日本語は □ になります。")
    if IN_COLAB:
        print("  → !apt-get -qq install fonts-ipafont-gothic を実行してランタイムを再起動すると直ります")

## 2. モデル

**CPU で動く大きさ**を選びます。層数がそのままレース距離になります（1層 = 100m）。

| モデル | 層数 | 距離 | メモリ(fp32) | CPU での1レース |
| --- | --- | --- | --- | --- |
| `llm-jp/llm-jp-3-150m` | 12 | 1200m | 約 0.7GB | 数秒 |
| `llm-jp/llm-jp-3-440m` | 16 | 1600m | 約 1.8GB | 十数秒 |
| `llm-jp/llm-jp-3-1.8b` | 24 | 2400m | 約 7GB | 1分前後 |

**既定は 150m** です。まず一通り通してから、余裕があれば大きいものに変えてください。
1.8b はメモリを 8GB 近く使うので、16GB の PC でぎりぎりです。

初回はモデルのダウンロードが入ります（150m で約 0.6GB）。
2 回目からは `~/.cache/huggingface` から読むだけなので速いです。

**このセルは 1 回だけ実行すれば十分**です。レースを作り直すときは 4 章から回してください。

In [ ]:
MODEL = "llm-jp/llm-jp-3-150m"   # 上の表から選ぶ
METERS_PER_LAYER = 100            # アプリ側の既定と合わせる

# GPU があれば使うが、無くてよい。mps（Apple Silicon）は層ごとの誤差が出ることがあるので既定は CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32)
except TypeError:
    # transformers 4.56 より前は dtype ではなく torch_dtype
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
model = model.to(DEVICE).eval()

n_layers = model.config.num_hidden_layers
print(f"{MODEL}  {n_layers}層 → {n_layers * METERS_PER_LAYER}m  ({DEVICE})")

## 3. logit lens

各層の隠れ状態を**最終層の LayerNorm と出力埋め込みに通して**、その層の時点での
「次の単語の確率」を見ます。層 0 は入力そのもので意味がないので 1 層目から使います。

`analyze()` は**プロンプトごとに 1 回だけ**推論して、語彙全体ぶんの確率を持ちます。
どの語を出走させるかはあとから選ぶので、推論をやり直す必要はありません。

In [ ]:
@torch.no_grad()
def analyze(prompt: str):
    """プロンプトを 1 回だけ流して、最終確率と層ごとの確率（語彙全体）を返す。

    戻り値: dict(final=[語彙], layers=[層, 語彙])
    語彙 10万・24層でも 10MB 程度なので、全部持ってしまってよい。
    """
    ids = {k: v.to(DEVICE) for k, v in tok(prompt, return_tensors="pt").items()}
    out = model(**ids, output_hidden_states=True)

    final = torch.softmax(out.logits[0, -1].float(), dim=-1).cpu()

    # 各層を出力空間へ射影する。norm の名前はモデルによって違うので拾いに行く
    base = model.model if hasattr(model, "model") else model.transformer
    norm = getattr(base, "norm", None) or getattr(base, "ln_f", None)
    head = model.get_output_embeddings()

    layers = []
    for h in out.hidden_states[1:]:            # 1層目から
        z = h[0, -1].float()
        if norm is not None:
            z = norm(z.to(next(norm.parameters()).dtype)).float()
        logits = head(z.to(head.weight.dtype)).float()
        layers.append(torch.softmax(logits, dim=-1).cpu())

    return {"prompt": prompt, "final": final, "layers": torch.stack(layers)}


def candidates(res, n=30):
    """最終確率の上位 n 語を一覧にする。ここから出走馬を選ぶ。"""
    top = torch.topk(res["final"], n)
    return pd.DataFrame({
        "no": range(n),
        "word": [tok.decode([i]) for i in top.indices.tolist()],
        "token_id": top.indices.tolist(),
        "prob": [round(float(v), 5) for v in top.values],
        "累積": [round(float(v), 4) for v in torch.cumsum(top.values, 0)],
    })

## 4. レースを1つ作る — ① 候補を見る

**ここから下を、レース1つにつき1回まわします。**

まずプロンプトを決めて、候補を一覧します。`no` の列が次のセルで使う番号です。
`累積` は上位からの確率の合計で、ここが 1.0 に近いところまでが「意味のある候補」です。

In [ ]:
RACE_NO = 1                 # レース番号。ファイル名 races_01.csv になる
PROMPT = "今日は"            # このレースのプロンプト
RACE_NAME = ""              # 空なら「第1R 「今日は」」が入る

res = analyze(PROMPT)
candidates(res, n=30)

## 5. レースを1つ作る — ② 出走馬を選ぶ

上の表の `no` を並べます。**8〜18 頭**にしてください（アプリ側の制限。
複勝・ワイド・3連系が3着まで数えるので 8 頭以上が要ります）。

選び方のこつ。

- **1番人気を外さない**。`no=0` を入れないと「本命不在」になって当てにくいだけのレースになります
- **確率が近いものを混ぜる**と接戦になります。逆に離れた語ばかりだと一本かぶりで面白くありません
- **同じ表層の語**（`▁今日` と `今日` など）は見た目が同じ馬になるので、どちらかにします
- 確率が極端に小さい語（1e-6 未満）は入れても走らないだけなので避けます

既定は上位 12 頭です。`PICK = [0, 1, 2, 3, 5, 8, 13, 21, ...]` のように書き換えてください。

In [ ]:
PICK = list(range(12))      # ← 上の表の no を並べる。8〜18 個

MIN_ENTRIES, MAX_ENTRIES = 8, 18   # アプリ側の制限

cand = candidates(res, n=max(PICK) + 1)
if len(PICK) != len(set(PICK)):
    raise ValueError("同じ番号が2回入っています。")
if not MIN_ENTRIES <= len(PICK) <= MAX_ENTRIES:
    raise ValueError(f"{len(PICK)} 頭は取り込めません（{MIN_ENTRIES}〜{MAX_ENTRIES} 頭）。")

sel = cand.loc[PICK]
words = sel.word.tolist()
token_ids = sel.token_id.tolist()
if len(set(words)) != len(words):
    print("⚠ 同じ表層の語が複数あります。見た目が同じ馬になります:",
          [w for w in set(words) if words.count(w) > 1])

print(f"{len(words)} 頭： " + " / ".join(f"{i+1}.{w}" for i, w in enumerate(words)))
sel

## 6. レースを1つ作る — ③ 順位変動を見る

**縦軸 logit** と**縦軸 probability** の 2 枚を出します。
順位がコロコロ入れ替わりすぎるレースはここで気づけるので、
プロンプトを変えるか、出走馬を選び直す判断ができます。

アプリ側でも平滑化は掛かりますが、**元データが暴れすぎているとレースになりません**。

In [ ]:
def race_probs(res, token_ids):
    """選んだ語だけを取り出す。戻り値: (最終確率 [k], 層ごとの確率 [k, 層数])"""
    idx = torch.tensor(token_ids)
    return res["final"][idx].numpy(), res["layers"][:, idx].numpy().T


def plot_race(words, probs, title=""):
    """probs: [頭, 層]"""
    T = probs.shape[1]
    x = np.arange(1, T + 1) * METERS_PER_LAYER
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for ax, (vals, name) in zip(axes, [(np.log(probs + 1e-12), "logit (log prob)"), (probs, "probability")]):
        for i, w in enumerate(words):
            ax.plot(x, vals[i], label=f"{i+1}.{w}", linewidth=1.6)
        ax.set_xlabel("distance [m]  (1 layer = %dm)" % METERS_PER_LAYER)
        ax.set_ylabel(name)
        ax.grid(alpha=.3)
    axes[1].legend(fontsize=8, ncol=2, loc="upper left")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


final_probs, layer_probs = race_probs(res, token_ids)
plot_race(words, layer_probs, f"R{RACE_NO}  「{PROMPT}」  {layer_probs.shape[1] * METERS_PER_LAYER}m")

## 7. レースを1つ作る — ④ 書き出し

オッズの初期値も入れます。手で 1 語ずつ入れるのは大変なので、確率から作ります。

```
oddsMean = clamp(0.80 / 最終確率, 1.1, 500)    ← 控除率 20% ぶんだけ甘い
oddsVar  = (0.18 × oddsMean)²                  ← 変動係数 18%
```

**上位人気が実際に来やすい**設定です。荒れさせたいレースだけ、アプリの GUI で分散を上げてください。

ファイルは `races_01.csv` のように**番号付き**で出ます。
アプリ側の取り込みは**追加**なので、作った順に何度取り込んでも前のレースは消えません。

In [ ]:
TAKEOUT = 0.80      # 単勝の控除率
CV = 0.18           # オッズの変動係数（大きいほど荒れる）

def initial_odds(p):
    mean = float(np.clip(TAKEOUT / max(p, 1e-4), 1.1, 500))
    return mean, (CV * mean) ** 2


race_id = f"R{RACE_NO:02d}"
rows = []
for i, w in enumerate(words):
    mean, var = initial_odds(float(final_probs[i]))
    row = {
        "race_id": race_id,
        "race_name": RACE_NAME or f"第{RACE_NO}R 「{PROMPT}」",
        "prompt": PROMPT,
        "model": MODEL,
        "word": w,
        "final_prob": float(final_probs[i]),
        "odds_mean": round(mean, 1),
        "odds_var": round(var, 2),
    }
    for t in range(layer_probs.shape[1]):
        row[f"layer_{t+1}"] = float(layer_probs[i, t])
    rows.append(row)

df = pd.DataFrame(rows)
OUT = f"races_{RACE_NO:02d}.csv"
df.to_csv(OUT, index=False, encoding="utf-8-sig")
print(f"{os.path.abspath(OUT)} に {len(df)} 頭 × {layer_probs.shape[1]} 層を書き出しました（{race_id}）")

made = sorted(glob.glob("races_*.csv"))
print("これまでに作ったレース:", ", ".join(made) if made else "なし")

if IN_COLAB:
    from google.colab import files
    files.download(OUT)

df.head()

## 次のレースを作る

4 章に戻って `RACE_NO` と `PROMPT` を変え、4 → 5 → 6 → 7 と実行するだけです。
モデルの読み込み（2 章）はやり直す必要がありません。

プロンプトの例。**続きが一意に決まらないもの**ほど接戦になって面白くなります。

| プロンプト | 傾向 |
| --- | --- |
| `今日は` | 助詞・副詞が競る。定番 |
| `大学の研究室で` | 動詞が割れる |
| `人工知能は` | 「人間」「私たち」など名詞戦 |
| `東京の天気は` | 「晴れ」「雨」で本命が立ちやすい |
| `この問題の答えは` | 記号や数字が混じる。荒れる |

## 補足：荒れ具合の調整

| やりたいこと | 触るところ |
| --- | --- |
| 人気どおりに決まりやすくする | `CV` を下げる（0.10 など） |
| 荒れさせる | `CV` を上げる（0.30 など）。または GUI で特定の語の分散だけ上げる |
| レースを長くする | 層の多いモデルにする。`METERS_PER_LAYER` は見た目の距離が変わるだけ |
| 接戦にする | 確率が近い語を選ぶ（5章） |

着順は**最終出力確率だけ**で決まります。途中経過をどういじっても結果は変わりません。